In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import seaborn as sns

## 2. Carregamento e Pré-processamento dos Dados

Importamos o CSV, renomeamos as colunas para português, removemos a coluna de direção do vento (não utilizada nesta análise) e convertemos a coluna de data/hora para o tipo `datetime`.

# Avaliação de Turbinas Eólicas
## Análise de Desempenho — Dataset T1 (2018)

Este projeto analisa os dados operacionais de uma turbina eólica ao longo do ano de 2018, comparando a potência ativa gerada com a curva de potência teórica. O objetivo é identificar quantos registros operam dentro dos limites aceitáveis de eficiência.

**Dataset:** 50.530 registros com medições a cada 10 minutos  
**Fonte:** `T1.csv`  
**Ferramentas:** Python · Pandas · Matplotlib · Seaborn

---

## 1. Importação de Bibliotecas

In [2]:
#Primeiro passo é importar os arquivos e verificar as colunas
turbina = pd.read_csv('T1.csv')

In [3]:
turbina.head()

,Date/Time,LV ActivePower (kW),Wind Speed (m/s),Theoretical_Power_Curve (KWh),Wind Direction (°)
0,01 01 2018 00:00,380.047791,5.311336,416.328908,259.994904
1,01 01 2018 00:10,453.769196,5.672167,519.917511,268.641113
2,01 01 2018 00:20,306.376587,5.216037,390.900016,272.564789
3,01 01 2018 00:30,419.645905,5.659674,516.127569,271.258087
4,01 01 2018 00:40,380.650696,5.577941,491.702972,265.674286


In [4]:
turbina.columns = ['Data/hora', 'ActivePower(kW)', 'WindSpeed(m/s)', 'Curva_Teórica(KWh)','Direção do Vento(°)']
del turbina['Direção do Vento(°)']
turbina['Data/hora']= pd.to_datetime(turbina['Data/hora'])
display(turbina)


,Data/hora,ActivePower(kW),WindSpeed(m/s),Curva_Teórica(KWh)
0,2018-01-01 00:00:00,380.047791,5.311336,416.328908
1,2018-01-01 00:10:00,453.769196,5.672167,519.917511
2,2018-01-01 00:20:00,306.376587,5.216037,390.900016
3,2018-01-01 00:30:00,419.645905,5.659674,516.127569
4,2018-01-01 00:40:00,380.650696,5.577941,491.702972
...,...,...,...,...
50525,2018-12-31 23:10:00,2963.980957,11.404030,3397.190793
50526,2018-12-31 23:20:00,1684.353027,7.332648,1173.055771
50527,2018-12-31 23:30:00,2201.106934,8.435358,1788.284755
50528,2018-12-31 23:40:00,2515.694092,9.421366,2418.382503


In [ ]:
# Informações gerais e estatísticas descritivas do dataset
print("=== Informações do Dataset ===")
turbina.info()
print("\n=== Estatísticas Descritivas ===")
turbina.describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=turbina, x='WindSpeed(m/s)', y='ActivePower(kW)', s=5, alpha=0.4, ax=ax)
ax.set_title('Potência Real × Velocidade do Vento')
ax.set_xlabel('Velocidade do Vento (m/s)')
ax.set_ylabel('Potência Ativa (kW)')
plt.tight_layout()
plt.show()

## 3. Visualização Exploratória

Nos gráficos abaixo comparamos a relação entre velocidade do vento e potência gerada — primeiro com os dados reais, depois com a curva teórica ideal. Espera-se que quanto maior a velocidade, maior a potência produzida.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=turbina, x='WindSpeed(m/s)', y='Curva_Teórica(KWh)', s=5, alpha=0.4, color='green', ax=ax)
ax.set_title('Curva de Potência Teórica × Velocidade do Vento')
ax.set_xlabel('Velocidade do Vento (m/s)')
ax.set_ylabel('Potência Teórica (KWh)')
plt.tight_layout()
plt.show()

## 4. Avaliação de Desempenho (Tolerância ±5%)

Para avaliar o desempenho da turbina, classificamos cada medição em três categorias com base na curva de potência teórica:

| Classificação | Critério |
|---|---|
| **Dentro** | Potência real dentro de ±5% da curva teórica |
| **Fora** | Potência real fora do intervalo aceitável |
| **Zero** | Geração igual a zero (parada ou vento insuficiente) |

In [10]:
#Criando valores de limite aceitáveis para as turbinas

pot_real= turbina['ActivePower(kW)'].tolist()
pot_teorica = turbina['Curva_Teórica(KWh)'].tolist()
pot_max=[]
pot_min=[]
dentro_limite = []

for potencia in pot_teorica:
    pot_max.append(potencia*1.05)
    pot_min.append(potencia*0.95)

for p, potencia in enumerate(pot_real):
    if potencia>=pot_min[p] and potencia<=pot_max[p]:
        dentro_limite.append('Dentro')
    elif potencia==0:
        dentro_limite.append('Zero')
    else:
        dentro_limite.append('Fora')
    

print(dentro_limite.count('Dentro')/len(dentro_limite))

0.37286760340391845


In [11]:
# Adicionando a lista "dentro do limite" ao df
turbina['DentroLimite'] = dentro_limite
display(turbina)

,Data/hora,ActivePower(kW),WindSpeed(m/s),Curva_Teórica(KWh),DentroLimite
0,2018-01-01 00:00:00,380.047791,5.311336,416.328908,Fora
1,2018-01-01 00:10:00,453.769196,5.672167,519.917511,Fora
2,2018-01-01 00:20:00,306.376587,5.216037,390.900016,Fora
3,2018-01-01 00:30:00,419.645905,5.659674,516.127569,Fora
4,2018-01-01 00:40:00,380.650696,5.577941,491.702972,Fora
...,...,...,...,...,...
50525,2018-12-31 23:10:00,2963.980957,11.404030,3397.190793,Fora
50526,2018-12-31 23:20:00,1684.353027,7.332648,1173.055771,Fora
50527,2018-12-31 23:30:00,2201.106934,8.435358,1788.284755,Fora
50528,2018-12-31 23:40:00,2515.694092,9.421366,2418.382503,Dentro


In [ ]:
# Distribuição percentual das classificações
total = len(turbina)
counts = turbina['DentroLimite'].value_counts()
percentual = (counts / total * 100).round(2)
resumo = pd.DataFrame({'Quantidade': counts, 'Percentual (%)': percentual})
print("=== Distribuição de Desempenho ===")
print(resumo)

# Gráfico de barras com percentuais
fig, ax = plt.subplots(figsize=(7, 4))
cores_bar = {'Dentro': 'steelblue', 'Fora': 'tomato', 'Zero': 'orange'}
bars = ax.bar(resumo.index, resumo['Quantidade'], color=[cores_bar[i] for i in resumo.index])
ax.set_title('Distribuição de Desempenho da Turbina (2018)')
ax.set_xlabel('Classificação')
ax.set_ylabel('Quantidade de Registros')
for i, (idx, row) in enumerate(resumo.iterrows()):
    ax.text(i, row['Quantidade'] + 200, f"{row['Percentual (%)']}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cores = {'Dentro': 'steelblue', 'Fora': 'tomato', 'Zero': 'orange'}
sns.scatterplot(data=turbina, x='WindSpeed(m/s)', y='ActivePower(kW)',
                hue='DentroLimite', s=4, palette=cores, ax=ax)
ax.set_title('Desempenho da Turbina: Potência Real vs. Limite ±5% da Curva Teórica')
ax.set_xlabel('Velocidade do Vento (m/s)')
ax.set_ylabel('Potência Ativa (kW)')
ax.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
# Análise mensal de eficiência (% de registros dentro do limite por mês)
turbina['Mes'] = turbina['Data/hora'].dt.month
eficiencia_mensal = turbina.groupby('Mes')['DentroLimite'].apply(
    lambda x: (x == 'Dentro').sum() / len(x) * 100
).round(2)

meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 13), eficiencia_mensal, color='steelblue', edgecolor='white')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses)
ax.set_title('Eficiência Mensal da Turbina — % dentro do limite ±5%')
ax.set_ylabel('% de Registros Dentro do Limite')
ax.set_ylim(0, 100)
media = eficiencia_mensal.mean()
ax.axhline(y=media, color='red', linestyle='--', label=f'Média anual: {media:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Conclusão

A análise dos dados operacionais da turbina eólica ao longo de **2018** (50.530 medições a cada 10 minutos) revelou os seguintes pontos:

### Resultados principais

- Apenas **~37% dos registros** estão dentro da tolerância de ±5% em relação à curva de potência teórica, indicando que a turbina opera frequentemente fora das condições ideais de eficiência.
- A **maioria dos registros** foi classificada como "Fora" do limite, concentrados especialmente em velocidades intermediárias de vento — sugerindo subutilização do potencial de geração nessas condições.
- Uma parcela dos registros apresenta geração **zero**, possivelmente associada a períodos de baixa velocidade do vento, paradas programadas ou manutenção.
- A análise mensal evidencia **variações sazonais** de eficiência, com alguns meses performando significativamente acima ou abaixo da média anual.

### Possíveis próximos passos

- Investigar as causas dos registros "Fora" do limite por faixa de velocidade do vento
- Correlacionar meses de baixa eficiência com dados de manutenção ou eventos climáticos
- Construir um modelo preditivo de geração com base na velocidade do vento (ex: regressão polinomial)
- Comparar com dados de outras turbinas do parque para identificar se o problema é localizado